### ii) The Cost-Basis Fragmentation Gap: 
Digital assets move seamlessly across chains. When an asset is received from an "unknown" sender (source), the cost basis is often lost. Cost Basis is simply the total price a user has paid to buy an asset, and an LP Token is a digital receipt for assets deposited into a trading pool. The POC here aims to look at that LP receipt to find the original buy prices (cost basis), so that an accurate claim of total profit (or loss) is conducted for regulatory reporting. 

To resolve this gap, the engine employs a **two-phase acquisition discovery** process. Phase 1 scans the most recent transaction history (based on the "Tx Limit") for inbound transfers matching each asset. If no acquisition is found within this window—common for long-held positions—the engine initiates a **Phase 2 targeted scan**, querying the blockchain indexer specifically for the earliest-ever inbound transfer of that particular token. This eliminates the need for expensive full-history scans whilst still recovering the original cost basis for assets acquired outside the initial audit window.

### iii) The "Regulatory Blindspot"
Current reporting frameworks like the OECD's CARF framework, which has a POC done in another notebook, requires specific metadata (transaction hash, fiat equivalent values). Generic blockchain explorers do not provide the high level aggregation required for institutional or individual tax disclosures. *This presents a big gap to business service and other firms attempting to provide a holistic view of a wallet (user's) activity*.

## 3. Technical Implementation & Dependencies
The framework relies on a modular "Adapter" architecture to ensure data integrity and real-time precision.

### 3.1 Core Dependencies:
There are several dependencies here that are required to run this successfully, these are:
*   **Alchemy API (Multi-Chain Indexer)**: Used as the primary gateway to the blockchain. While many tools use basic RPC nodes (Remote Procedure Call Nodes) which are the bridge that lets Jupyter notebook interact with the blockchain itself. Alchemy's `getTokenBalances` and `getAssetTransfers` methods provide optimised indexing that allows for sub-second balance retrieval across chains like Ethereum, Polygon, and Base.

*   **Moralis API (Historical Price Engine)**: Essential for P&L calculations. It allows the tool to query the price of a specific token at a specific **Block Number**, rather than a generic timestamp. *This is critical for defending audit positions where price volatility can change 5-10% within a single hour*.

*   **Dexscreener API**: Provides real-time pricing and 24h/1w/1m changes for "Long Tail" assets that are not yet listed on major centralized exchanges (CEXs). This can prove an issue for more obscure assets present on blockchains.

*   **IPyWidgets & IPython**: Enables the GUI (graphical user interfaces) experience. This transforms static Python code into a dynamic dashboard accessible to non-technical auditors.

*   **Pandas & NumPy**: Used for vectorised performance calculations (ROI, Value-at-Risk) and styling the final compliance ledger.

*   **OS & Dotenv**: Used to securely bridge the notebook with the users computer's system to load private API keys from a hidden "vault" (.env file) without exposing them in the code.

*   **Requests & JSON**: Functions as the "digital courier" that sends orders to the blockchain indexers and unwraps the resulting data packages into a format the notebook can read.

*   **Time & RE**: Responsible for managing the rate of requests of the audit to prevent API overloading and scanning long strings of data for specific wallet address patterns.

*   **Base64**: Acts as a digital translator that converts raw data into a unique string format, enabling users to download the final audit results directly to their browser.

*   **ABC & Typing**: Provides a set of blueprints and labels that ensure every part of the software is built consistently and handles data types like lists and dictionaries correctly.


In [1]:
import os, re, requests, json, time, pandas as pd, base64
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets
from abc import ABC, abstractmethod
from typing import List, Dict, Any
from dotenv import load_dotenv

load_dotenv()
MORALIS_KEY = os.getenv("MORALIS_API_KEY") or (st.secrets.get("MORALIS_API_KEY") if hasattr(st, "secrets") else "")
ALCHEMY_KEY = os.getenv("ALCHEMY_API_KEY") or (st.secrets.get("ALCHEMY_API_KEY") if hasattr(st, "secrets") else "")

NATIVE_WRAPPERS = {
    "ETHEREUM": "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2",  # WETH (for Moralis)
    "POLYGON": "0x7d1afa7b718fb893db30a3abc0cfc608aacfebb0",  # MATIC ERC-20 (for Moralis)
    "BASE": "0x4200000000000000000000000000000000000006"       # WETH on Base (for Moralis)
}

# Separate wrapper addresses used for Dexscreener fallback (on-chain addresses per network)
DEXSCREENER_NATIVE_WRAPPERS = {
    "ETHEREUM": "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2",  # WETH on Ethereum
    "POLYGON": "0x0d500b1d8e8ef31e21c99d1db9a6444d3adf1270",  # WMATIC on Polygon
    "BASE": "0x4200000000000000000000000000000000000006"       # WETH on Base
}

import functools

@functools.lru_cache(maxsize=1024)
def get_historical_price_moralis(address, chain, block=None):
    if not MORALIS_KEY: return 0
    m_chain = {"ETHEREUM": "eth", "POLYGON": "polygon", "BASE": "base"}.get(chain.upper(), "eth")
    url = f"https://deep-index.moralis.io/api/v2.2/erc20/{address}/price?chain={m_chain}"
    if block: url += f"&to_block={block}"
    headers = {"accept": "application/json", "X-API-Key": MORALIS_KEY}
    try:
        r = requests.get(url, headers=headers, timeout=10).json()
        return float(r.get("usdPrice", 0))
    except: return 0

@functools.lru_cache(maxsize=32)
def get_latest_block(chain):
    url = {"ETHEREUM": f"https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_KEY}",
           "POLYGON": f"https://polygon-mainnet.g.alchemy.com/v2/{ALCHEMY_KEY}",
           "BASE": f"https://base-mainnet.g.alchemy.com/v2/{ALCHEMY_KEY}"}.get(chain.upper())
    if not url: return 0
    try:
        r = requests.post(url, json={"id":1, "jsonrpc":"2.0", "method":"eth_blockNumber"}).json()
        return int(r.get("result", "0x0"), 16)
    except: return 0

def get_dexscreener_price(address, chain):
    """Fallback price discovery via Dexscreener (free, no API key)."""
    chain_map = {"ETHEREUM": "ethereum", "POLYGON": "polygon", "BASE": "base"}
    ds_chain = chain_map.get(chain.upper(), "ethereum")
    try:
        url = f"https://api.dexscreener.com/tokens/v1/{ds_chain}/{address}"
        resp = requests.get(url, timeout=10).json()
        if isinstance(resp, list) and len(resp) > 0:
            # Pick the pair with highest liquidity
            best = max(resp, key=lambda p: float(p.get("liquidity", {}).get("usd", 0) or 0))
            return float(best.get("priceUsd", 0) or 0)
        return 0
    except:
        return 0

class AuditResult:
    def __init__(self, chain, symbol, balance, address):
        self.chain = chain; self.symbol = symbol; self.balance = balance; self.address = address
    def to_dict(self): return {"Chain": self.chain.upper(), "Symbol": self.symbol, "Balance": self.balance, "Address": self.address}

class AlchemyAdapter:
    def __init__(self, api_key):
        self.api_key = api_key
        self.urls = {"ethereum": f"https://eth-mainnet.g.alchemy.com/v2/{api_key}",
                     "polygon": f"https://polygon-mainnet.g.alchemy.com/v2/{api_key}",
                     "base": f"https://base-mainnet.g.alchemy.com/v2/{api_key}"}

    def get_gas_fee_usd(self, chain, tx_hash, block):
        url = self.urls.get(chain.lower())
        if not url: return 0
        try:
            p_rec = {"id":1,"jsonrpc":"2.0","method":"eth_getTransactionReceipt","params":[tx_hash]}
            rec = requests.post(url, json=p_rec).json().get("result", {})
            gas_used = int(rec.get("gasUsed", "0x0"), 16)
            p_tx = {"id":1,"jsonrpc":"2.0","method":"eth_getTransactionByHash","params":[tx_hash]}
            tx = requests.post(url, json=p_tx).json().get("result", {})
            gas_price = int(tx.get("gasPrice", "0x0"), 16)
            fee_native = (gas_used * gas_price) / 1e18
            native_addr = NATIVE_WRAPPERS.get(chain.upper(), NATIVE_WRAPPERS["ETHEREUM"])
            native_price = get_historical_price_moralis(native_addr, chain, block)
            return fee_native * native_price
        except: return 0

    def get_positions(self, address):
        res = []
        for name, url in self.urls.items():
            try:
                p_eth = {"id":1,"jsonrpc":"2.0","method":"eth_getBalance","params":[address,"latest"]}
                resp = requests.post(url, json=p_eth, timeout=10).json()
                val = int(resp.get("result","0x0"),16)/1e18
                if val > 1e-6: 
                    res.append(AuditResult(name, "NATIVE", val, "native"))
                p_t = {"id":1,"jsonrpc":"2.0","method":"alchemy_getTokenBalances","params":[address]}
                resp_t = requests.post(url, json=p_t, timeout=10).json()
                for b in resp_t.get("result",{}).get("tokenBalances",[]):
                    bal = int(b["tokenBalance"],16)
                    if bal > 0:
                        contract_addr = b["contractAddress"]
                        try:
                            meta_payload = {"id":1,"jsonrpc":"2.0","method":"alchemy_getTokenMetadata","params":[contract_addr]}
                            meta = requests.post(url, json=meta_payload, timeout=10).json().get("result", {})
                            symbol = str(meta.get("symbol") or "TOKEN")
                            # Robust decimal parsing
                            try:
                                decimals = int(meta.get("decimals", 18))
                            except (TypeError, ValueError):
                                decimals = 18
                        except Exception:
                            symbol, decimals = "TOKEN", 18
                        true_bal = bal / (10 ** decimals)
                        res.append(AuditResult(name, symbol, true_bal, contract_addr))
            except Exception as e:
                st.sidebar.error(f"Error on {name}: {e}")
        return res

    def targeted_acquisition_scan(self, wallet, contract_addr, chain):
        """Targeted scan: find the EARLIEST inbound transfer of a specific token to this wallet."""
        url = self.urls.get(chain.lower())
        if not url: return None
        try:
            p = {"id":1,"jsonrpc":"2.0","method":"alchemy_getAssetTransfers","params":[{
                "toAddress": wallet,
                "contractAddresses": [contract_addr],
                "category": ["erc20"],
                "order": "asc",
                "maxCount": "0x1",
                "withMetadata": True
            }]}
            transfers = requests.post(url, json=p, timeout=10).json().get("result", {}).get("transfers", [])
            if transfers:
                tx = transfers[0]
                ts = tx.get("metadata", {}).get("blockTimestamp", "Unknown")
                return {"Hash": tx["hash"], "Block": int(tx["blockNum"], 16), "Value": tx.get("value", 0), "Timestamp": ts}
        except:
            pass
        return None

    def targeted_native_scan(self, wallet, chain):
        """Targeted scan: find the EARLIEST inbound native (ETH/POL) transfer to this wallet."""
        url = self.urls.get(chain.lower())
        if not url: return None
        try:
            p = {"id":1,"jsonrpc":"2.0","method":"alchemy_getAssetTransfers","params":[{
                "toAddress": wallet,
                "category": ["external"],
                "order": "asc",
                "maxCount": "0x1",
                "withMetadata": True
            }]}
            transfers = requests.post(url, json=p, timeout=10).json().get("result", {}).get("transfers", [])
            if transfers:
                tx = transfers[0]
                ts = tx.get("metadata", {}).get("blockTimestamp", "Unknown")
                return {"Hash": tx["hash"], "Block": int(tx["blockNum"], 16), "Value": tx.get("value", 0), "Timestamp": ts}
        except:
            pass
        return None

    def get_recent_transactions(self, address, limit):
        txs = []
        seen_hashes = set()
        for name, url in self.urls.items():
            for direction in [{"fromAddress": address}, {"toAddress": address}]:
                p = {"id":1,"jsonrpc":"2.0","method":"alchemy_getAssetTransfers","params":[{
                    **direction, "maxCount":f"0x{int(limit):x}", "category":["external","erc20"], "withMetadata":True
                }]}
                try:
                    for tx in requests.post(url, json=p).json().get("result",{}).get("transfers",[]):
                        if tx["hash"] in seen_hashes: continue
                        seen_hashes.add(tx["hash"])
                        raw = tx.get("rawContract", {})
                        contract = raw.get("address", "native") if raw else "native"
                        ts = tx.get("metadata", {}).get("blockTimestamp", "Unknown")
                        txs.append({"Chain":name.upper(), "Hash":tx["hash"], "From":tx.get("from"),
                                    "Asset":tx.get("asset"), "Value":tx.get("value"), 
                                    "Block":int(tx["blockNum"],16), "To":tx["to"], "Contract": contract,
                                    "Timestamp": ts})
                except: pass
        return txs

class UniversalAuditEngine:
    def __init__(self, key): self.adapter = AlchemyAdapter(key)
    def run_audit(self, addr): return [p.to_dict() for p in self.adapter.get_positions(addr)]
    def get_recent_history(self, addr, lim): return self.adapter.get_recent_transactions(addr, lim)
    def get_gas_cost(self, chain, h, b): return self.adapter.get_gas_fee_usd(chain, h, b)
    def targeted_token_scan(self, wallet, contract, chain): return self.adapter.targeted_acquisition_scan(wallet, contract, chain)
    def targeted_native(self, wallet, chain): return self.adapter.targeted_native_scan(wallet, chain)

print('\u2705 Section 1: Core Models & Real-Time Adapters Loaded.')


### 3.2 Advanced GUI for P&L Calculations And Analysis:

This section provides a user-friendly Control Dashboard that allows the user to interact with the audit logic without writing any code. Through this GUI, the user can input specific wallet addresses, set transaction history limits (to control data depth), and toggle between different performance timeframes (24h, 1w, or 1m) to see how the portfolio has evolved across time.

In [ ]:
wallet_input = widgets.Text(value="0x28c6c06298d514db089934071355e5743bf21d60", description='Target Wallet:', layout=widgets.Layout(width='600px'), style={'description_width': 'initial'})
limit_input = widgets.IntSlider(value=10, min=1, max=10, description='Tx Limit:', layout=widgets.Layout(width='500px'))
timeframe_input = widgets.Dropdown(options=['24h', '1w', '1m'], value='24h', description='Timeframe:', layout=widgets.Layout(width='500px'))
scope_btn = widgets.Button(description='\U0001f50d Run Advanced Portfolio Audit', button_style='primary', layout=widgets.Layout(width='300px', height='40px'))
output_area = widgets.Output()

display(widgets.VBox([wallet_input, limit_input, timeframe_input, scope_btn, output_area]))
print('\u2705 Section 2: Advanced Scope GUI Ready.')


✅ Section 2: Advanced Scope GUI Ready.


## 3.3 USD Reconciliation (Public Price Discovery):

In this section, the tool performs "Price Discovery" by connecting to external pricing engines (Moralis and Dexscreener). The tool translates the raw number of tokens on the blockchain into their equivalent USD market value at either the current moment or a specific historical block height. This ensures that the reconciliation is based on real-world "Fair Market Value" rather than arbitrary estimates.

### 3.3.1 Spam Detection & Reconciliation Integrity
In the decentralised ecosystem, wallets frequently receive unsolicited "airdrop" tokens from malicious actors. These tokens — often branded with recognisable names such as luxury brands or popular meme coins — are designed to lure wallet holders into interacting with malicious smart contracts, which can result in the loss of legitimate assets. From a reconciliation standpoint, these tokens present a data integrity challenge: they appear as valid ERC-20 balances on-chain but have no genuine market value, no liquidity pool, and no trading history on any decentralised exchange.

This tool addresses this through a **multi-source price verification engine**. Each non-native token is queried against two independent pricing APIs — Moralis (centralised index) and Dexscreener (decentralised exchange aggregator). If both sources return a price of $0.00, the asset is programmatically flagged as **"Probable Spam (No Market)"** with a value of $0.00. This heuristic is market-data driven rather than rule-based (i.e., it does not rely on fragile name-matching against known brands), ensuring that the classification is both scalable and defensible under audit scrutiny.

Importantly, assets flagged as probable spam are **excluded from all aggregate performance metrics** — they do not inflate Portfolio Value, nor do they dilute or skew the ROI calculation. This ensures that the headline figures reflect only assets with verifiable market participation, maintaining the institutional-grade integrity required under CARF and MiCA reporting frameworks.

In [3]:
def on_run_audit(b):
    with output_area:
        output_area.clear_output()
        engine = UniversalAuditEngine(ALCHEMY_KEY)
        tf = tf.value
        wallet = wallet.value
        print(f"Auditing {wallet[:10]}... [Timeframe: {tf}]")
        _audit_start = time.time()
        
        data = engine.run_audit(wallet)
        tx_data = engine.get_recent_history(wallet, limit_input.value)
        
from concurrent.futures import ThreadPoolExecutor

        def process_pos(pos):
            chain = pos["Chain"]
            addr = pos["Address"]
            is_native = (addr == "native" or addr is None)
            
            # Add Owner Wallet for clarity and CARF reporting
            pos["Owner Wallet"] = wallet
            
            latest_block = get_latest_block(chain)
            block_offsets = {
                '24h': 7200, '1w': 50400, '1m': 216000,
                '3m': 648000, '6m': 1296000, '1y': 2628000,
                '5y': 13140000, '10y': 26280000, 'All Time': 99999999
            }
            hist_block = max(1, latest_block - block_offsets.get(tf, 7200))
            
            # --- PRICE DISCOVERY (Dexscreener fallback → Moralis) ---
            price_source = "Unknown"
            if is_native:
                wrapper = NATIVE_WRAPPERS.get(chain.upper(), NATIVE_WRAPPERS["ETHEREUM"])
                ds_wrapper = DEXSCREENER_NATIVE_WRAPPERS.get(chain.upper(), DEXSCREENER_NATIVE_WRAPPERS["ETHEREUM"])
                curr_price = get_dexscreener_price(ds_wrapper, chain)
                if curr_price > 0:
                    price_source = "Dexscreener (Live DEX)"
                else:
                    curr_price = get_historical_price_moralis(wrapper, chain)
                    if curr_price > 0:
                        price_source = "Moralis (Block-Precise)"
                    else:
                        price_source = "Unpriced (No Market)"
                hist_price = get_historical_price_moralis(wrapper, chain, hist_block) or curr_price
            else:
                curr_price = get_dexscreener_price(addr, chain)
                if curr_price > 0:
                    price_source = "Dexscreener (Live DEX)"
                else:
                    curr_price = get_historical_price_moralis(addr, chain)
                    if curr_price > 0:
                        price_source = "Moralis (Block-Precise)"
                hist_price = get_historical_price_moralis(addr, chain, hist_block)
            
            # --- COST BASIS DISCOVERY (Broad scan → Targeted scan) ---
            unit_cost = 0; gas_cost = 0
            cost_basis_source = "Unknown"
            s_addr = str(addr or "").lower()
            s_wall = str(wallet or "").lower()
            
            # Phase 1: Check broad transaction history
            if is_native:
                asset_txs = [t for t in tx_data if str(t.get('Contract', '')).lower() == 'native' and str(t.get('To', '')).lower() == s_wall]
            else:
                asset_txs = [t for t in tx_data if str(t.get('Contract', '')).lower() == s_addr and str(t.get('To', '')).lower() == s_wall]
            
            acquisition_found = False
            acq_hash = "N/A"
            acq_date = "N/A"
            acq_amount = "N/A"
            if asset_txs:
                # Weighted Average Cost (WAC) Execution
                total_usd_spent = 0
                total_gas_spent = 0
                total_tokens_acquired = 0
                price_addr = wrapper if is_native else addr
                
                sorted_txs = sorted(asset_txs, key=lambda x: x['Block'])
                first_tx = sorted_txs[0]
                
                for tx in sorted_txs:
                    tx_val = float(tx.get('Value', 0) or 0)
                    if tx_val > 0:
                        tx_price = get_historical_price_moralis(price_addr, chain, tx['Block'])
                        tx_gas = engine.get_gas_cost(chain, tx['Hash'], tx['Block'])
                        total_usd_spent += (tx_price * tx_val)
                        total_gas_spent += tx_gas
                        total_tokens_acquired += tx_val

                if total_tokens_acquired > 0:
                    unit_cost = total_usd_spent / total_tokens_acquired
                    gas_cost = total_gas_spent
                    acq_amount = total_tokens_acquired
                else:
                    unit_cost = get_historical_price_moralis(price_addr, chain, first_tx['Block'])
                    gas_cost = engine.get_gas_cost(chain, first_tx['Hash'], first_tx['Block'])
                    acq_amount = float(first_tx.get('Value', 0) or 0)

                acquisition_found = True
                cost_basis_source = "Tx History (Phase 1 - WAC)"
                acq_hash = first_tx['Hash']  # Anchor hash to oldest known entry point
                acq_date = first_tx.get('Timestamp', 'Unknown')
            else:
                # Phase 2: Targeted scan for earliest inbound transfer
                if is_native:
                    targeted = engine.targeted_native(target_wallet, chain)
                else:
                    targeted = engine.targeted_token_scan(target_wallet, addr, chain)
                
                if targeted:
                    price_addr = wrapper if is_native else addr
                    unit_cost = get_historical_price_moralis(price_addr, chain, targeted['Block'])
                    gas_cost = engine.get_gas_cost(chain, targeted['Hash'], targeted['Block'])
                    acquisition_found = True
                    cost_basis_source = "Targeted Scan (Phase 2)"
                    acq_hash = targeted['Hash']
                    acq_date = targeted.get('Timestamp', 'Unknown')
                    acq_amount = float(targeted.get('Value', 0) or 0)
            
            total_pos_cost = (unit_cost * pos['Balance']) + gas_cost
            total_pos_value = pos["Balance"] * curr_price
            
            # --- AUDIT STATUS & SPAM DETECTION ---
            pos["_raw_total_pos_value"] = total_pos_value
            pos["_raw_total_pos_cost"] = total_pos_cost
            pos["_raw_acquisition_found"] = acquisition_found
            
            if curr_price == 0 and not is_native:
                # Both Moralis AND Dexscreener returned $0 → Probable Spam
                pos["Audit Status"] = "🚫 Probable Spam (No Market)"
                pos["Current Price (USD)"] = "$0.00 (Unpriced)"
                pos["Value (USD)"] = "$0.00"
                pos[f"Change ({tf}) (%)"] = "N/A"
                pos["Gas Expense (USD)"] = "N/A"
                pos["Overall ROI (%)"] = "N/A"
                pos["Price Source"] = "None (Probable Spam)"
                pos["Cost Basis Source"] = "N/A"
                pos["Cost Basis Unit Price (USD)"] = "N/A"
                pos["Acquisition Amount"] = "N/A"
                pos["Acquisition TX Hash"] = "N/A"
                pos["Acquisition Date (UTC)"] = "N/A"
            else:
                # Legitimate asset — set audit status
                if acquisition_found:
                    pos["Audit Status"] = "✅ Verified"
                elif pos["Balance"] > 0:
                    pos["Audit Status"] = "⚠️ No Acquisition Found"
                else:
                    pos["Audit Status"] = "✅ Verified"
                
                # Price & Value
                pos["Current Price (USD)"] = round(curr_price, 2)
                pos["Value (USD)"] = round(total_pos_value, 2)
                pos["Price Source"] = price_source
                pos["Cost Basis Source"] = cost_basis_source if acquisition_found else "Not Found"
                pos["Cost Basis Unit Price (USD)"] = round(unit_cost, 2) if acquisition_found else "N/A"
                pos["Acquisition Amount"] = round(acq_amount, 6) if isinstance(acq_amount, (int, float)) else acq_amount
                pos["Acquisition TX Hash"] = acq_hash
                pos["Acquisition Date (UTC)"] = acq_date
                
                # Change calculation
                if hist_price > 0:
                    pos[f"Change ({tf}) (%)"] = round(((curr_price - hist_price)/hist_price*100), 2)
                else:
                    pos[f"Change ({tf}) (%)"] = "N/A"
                
                # ROI Logic
                if acquisition_found:
                    pos["Gas Expense (USD)"] = round(gas_cost, 2)
                    pos["Overall ROI (%)"] = round(((total_pos_value - total_pos_cost)/total_pos_cost*100) if total_pos_cost > 0 else 0, 2)
                else:
                    pos["Gas Expense (USD)"] = "N/A"
                    pos["Overall ROI (%)"] = "N/A (No Acquisition Found)"
            
            # Round balance for display
            pos["Balance"] = round(pos["Balance"], 6)

        # Execute Parallel Processing (Limit to 5 workers to stay under Moralis rate limit safely)
        with ThreadPoolExecutor(max_workers=5) as executor:
            list(executor.map(process_pos, data))
            
        # Thread-safe aggregations post-map
        total_value = sum(p.get("_raw_total_pos_value", 0) for p in data)
        total_verified_value = sum(p.get("_raw_total_pos_value", 0) for p in data if p.get("_raw_acquisition_found", False))
        total_gain = sum((p.get("_raw_total_pos_value", 0) - p.get("_raw_total_pos_cost", 0)) for p in data if p.get("_raw_acquisition_found", False) and p.get("_raw_total_pos_cost", 0) > 0)
        
        # Cleanup temp vars
        for p in data:
            p.pop("_raw_total_pos_value", None)
            p.pop("_raw_total_pos_cost", None)
            p.pop("_raw_acquisition_found", None)
            
        df_final = pd.DataFrame(data)
        _audit_elapsed = time.time() - _audit_start
        _fmt = (f"{int(_audit_elapsed//60)}m {_audit_elapsed%60:.1f}s"
                if _audit_elapsed >= 60 else f"{_audit_elapsed:.1f}s")

        _audit_elapsed = time.time() - _audit_start
        _fmt = (f"{int(_audit_elapsed//60)}m {_audit_elapsed%60:.1f}s"
                if _audit_elapsed >= 60 else f"{_audit_elapsed:.1f}s")
                
        global df_final, global_tx_data
        df_final = pd.DataFrame(data); global_tx_data = tx_data
        
        avg_roi = (total_gain / total_verified_value * 100) if total_verified_value > 0 else 0
        header = f"<div style='display:flex; gap:20px;'><div style='padding:15px; background:#f8f9fa0a; border-radius:10px;'>" \
                 f"<small>Portfolio Value</small><h3>${total_value:,.2f}</h3></div>" \
                 f"<div style='padding:15px; background:#f8f9fa0a; border-radius:10px;'>" \
                 f"<small>Performance (ROI Incl. Gas)</small><h3 style='color:{'#00ffa3' if avg_roi>=0 else '#ff4b4b'};'>{avg_roi:+.2f}%</h3></div>" \
                 f"<div style='padding:15px; background:#f8f9fa0a; border-radius:10px;'>" \
                 f"<small>⏱️ Audit Duration</small><h3>{_fmt}</h3></div></div>"
                 
        display(HTML(header + "<br>"))
        display(df_final)

scope_btn.on_click(on_run_audit)
print('\u2705 Section 3: Async Real-Time P&L Engine Ready.')


## 4. Compliance & Risk Audit
This section implements an automated Risk Evaluation Engine that scans the transaction history for patterns that trigger regulatory concern under the CARF framework. Specifically, the engine focuses on:
i) High-Value Transfers: Identifying individual transactions exceeding 50 tokens, which may trigger *"Enhanced Due Diligence" (EDD)* reporting requirements.
ii) Self-Transfer Detection: Flagging Round-tripping (where assets are moved between different wallets owned by the same user), a common pattern used in tax-loss harvesting or wash trading which requires specific disclosure.
iii) Audit Trail Verification: Creating a permanent, time-stamped log of risks identified during the audit to ensure transparency and simplified documentation for authorities.

By automating such checks, this tool removes the manual burden of checking every transaction hash, providing an immediate *"Risk Summary"* for the entire multi-chain portfolio. 

In [4]:
def run_compliance_report(txs, wallet_address):
    if not txs: return "No transaction data available."
    risk_flags = []
    wallet_lower = str(wallet_address).lower().strip()
    
    # PASS 1: Map outbounds (disposals)
    out_map = {}
    for tx in txs:
        if str(tx.get('From', '')).lower() == wallet_lower:
            a = str(tx.get('Asset', '')).upper()
            v = float(tx.get('Value', 0) or 0)
            if a not in out_map: out_map[a] = []
            out_map[a].append(v)
            
    # PASS 2: Multi-Pass Detection
    for tx in txs:
        val = float(tx.get('Value', 0) or 0)
        asset = str(tx.get('Asset', '')).upper()
        h = tx['Hash']
        f, t = str(tx.get('From', '')).lower(), str(tx.get('To', '')).lower()
        
        # 1. High Value EDD
        if val > 50: risk_flags.append({"Hash": h, "Risk": "Medium", "Type": "CARF-EDD", "Issue": f"High Value Transfer ({val} {asset})"})
        # 2. Round-tripping
        if f == t and f == wallet_lower: risk_flags.append({"Hash": h, "Risk": "High", "Type": "Tax-Risk", "Issue": "Self-Transfer / Round-trip"})
        # 3. Bridge Activity
        if any(k in asset for k in ["BRIDGE", "HOP", "STARGATE", "ACROSS", "SYNAPSE"]):
            risk_flags.append({"Hash": h, "Risk": "Low", "Type": "Cross-Chain", "Issue": f"Bridge Detected ({asset})"})
        # 4. Wash Trading (Re-acquisition match)
        if t == wallet_lower and asset in out_map:
            if any(abs(ov - val) < 0.0001 for ov in out_map[asset]):
                risk_flags.append({"Hash": h, "Risk": "High", "Type": "Market-Integrity", "Issue": f"Potential Wash Trade ({asset})"})
    
    if not risk_flags: return "✅ No critical compliance risks detected."
    return pd.DataFrame(risk_flags).drop_duplicates()

def on_run_compliance(b):
    with compliance_output:
        compliance_output.clear_output()
        if 'global_tx_data' in globals() and global_tx_data:
            display(HTML("<h4>🛡️ CARF Compliance & Risk Summary</h4>"))
            report = run_compliance_report(global_tx_data, wallet_input.value)
            if isinstance(report, pd.DataFrame): display(report)
            else: print(report)
        else:
            print("⚠️ Please run the Audit first!")

✅ Section 4: Compliance Reporting Engine Loaded.


## 5. Export Audit Results:

The final module provides a Secure Data Export feature. Once the audit and compliance checks are complete, the user can generate a professional CSV ledger with a single click. This feature uses digital encoding to create a direct download link into the user's  browser, allowing them to save the audit results for their records or to share with relevant professionals and/or regulatory bodies as required.

In [5]:
def on_export(b):
    if 'df_final' in globals():
        csv = df_final.to_csv(index=False)
        b64 = base64.b64encode(csv.encode()).decode()
        filename = f"audit_{int(time.time())}.csv"
        href = f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="padding:10px; background:#198754; color:white; border-radius:5px;">\U0001f4be Download Report</a>'
        with export_output: 
            export_output.clear_output()
            display(HTML(href))
    else: print("\u26a0\ufe0f No data to export!")

export_btn = widgets.Button(description='\U0001f4be Prepare Browser Download', button_style='success', layout=widgets.Layout(width='300px'))
export_output = widgets.Output()
export_btn.on_click(on_export)
display(widgets.VBox([export_btn, export_output]))
print('\u2705 Section 5: Export functionality loaded.')


✅ Section 5: Export functionality loaded.


## 6. Current Limitations:
While this POC provides a robust framework for DeFi reconciliation, certain technical constraints remain:

i) Historical Data Depth: The tool currently scans the most recent transaction history based on the "Tx Limit" slider. For wallets with thousands of transactions, reaching the "true" initial cost basis may require increasing API pagination depth.

ii)Concentrated Liquidity (Uniswap V3): Standard ERC-20 LP tokens are fully supported; however, Uniswap V3 positions (which are represented as NFTs) require a specialized decomposition engine to calculate the underlying asset ratios.

iii) Gas Fee Approximation: The gas accounting logic assumes the first detected inbound transfer is the primary acquisition point. If an asset was acquired via multiple small buys, the gas cost is currently anchored to the most significant initial entry.

iv) API Rate Limits: As a browser-based tool, performance is subject to the rate limits of the free-tier API keys provided (Alchemy, Moralis, and Dexscreener).

v) Token Decimal Precision: The current implementation assumes all ERC-20 tokens use 18 decimal places when converting raw blockchain balances to human readable values. Tokens with non-standard decimals (e.g., USDC uses 6, WBTC uses 8) may display inflated balances, which cascades into incorrect Value (USD) and ROI calculations. *A production-grade solution would require an additional decimals() RPC call per contract to dynamically normalise each token's balance.*

## 6.1 Future exploration:
To evolve this framework into a production-grade audit suite, the following areas could be explored:

i) Multi-Wallet Aggregation: Implementing "Combined Entity" reporting where a user can link multiple hardware and software wallets to see a consolidated CARF risk report.

ii) Tax-Loss Harvesting Engine: Adding a dedicated dashboard that flags assets currently trading below their calculated cost basis, identifying opportunities for strategic tax-loss harvesting before the end of the fiscal year.

iii) Cross-Chain Bridge Reconciliation: Deepening the logic to "trace" assets as they move across bridges (e.g., from Ethereum to L2s like Base) to maintain a continuous cost-basis chain.

iv) Automated Tax Form Generation: Expanding the "Export" feature to directly populate standardized tax forms (like IRS Form 8949 or local equivalents) based on the reconciled capital gains data.

## 7. Conclusion:

The Liquidity Pool (LP) Reconciliation tool demonstrates that the "Regulatory Blindspot" in decentralised finance is not a permanent fixture, but a technical hurdle that can be overcome through automated indexing and real-time price discovery. By successfully unbundling complex LP tokens into their underlying assets and establishing a verifiable cost basis through historical block-height analysis, this POC provides a scalable blueprint for institutional-grade auditing better understanding financial metrics.

Beyond simple compliance, the ability to decompose multi-chain data has profound implications for financial analysis:

i) Precision in P&L: By anchoring calculations to specific block numbers via the Moralis API, the tool moves past estimated values to establish a definitive, audit-ready Profit & Loss statement that can withstand regulatory scrutiny.

ii) True ROI (return on investment) Discovery: Establishing an accurate cost basis for assets transferred from external sources allows for the calculation of Lifetime ROI, providing users and firms with a realistic view of investment performance that accounts for entry-point volatility and gas fees.

iii) Dynamic Valuation: Integrating real time "Fair Market Value" through Dexscreener and Moralis enables high-fidelity valuation of "long-tail" DeFi assets, which are often mispriced or invisible on traditional financial dashboards (DefiLlama, 2026).

As global frameworks such as OECD's CARF and Europe's MiCA transition from policy to enforcement, the ability to provide a *holistic view* of multi-chain activity will become a mandatory requirement for business service firms and individual investors alike. This tool bridges that critical gap, transforming raw, fragmented blockchain data into a structured, transparent, and audit-ready ledger. Future iterations of this framework will continue to refine risk evaluation models, ensuring that compliance remains programmable, precise, and permanent (Zetzsche *et al.*, 2020).


## 8. Bibliography:

*   **Bank for International Settlements (BIS) (2023).** *DeFi: Ecosystem, Risks and Options for Regulation.* Monetary and Economic Department.
*   **Chainalysis (2023).** *The 2023 Geography of Cryptocurrency Report.* [Patterns of cross-border DeFi flow and self-custody risk].
*   **DefiLlama (2026).** *Total Value Locked and Protocol Analytics.* [On-chain attribution data for LP reconciliation].
*   **European Commission (2023).** *Markets in Crypto-Assets Regulation (MiCA).* [Structural requirements for asset service providers].
*   **HMRC (2024).** *Cryptoassets Manual: Compliance and Reporting.* [UK specific tax treatment for DeFi and Staking].
*   **IOSCO (2023).** *Policy Recommendations for Decentralized Finance (DeFi).* Final Report FR08/23.
*   **Messari Crypto (2024).** *State of DeFi: Q1 2024 Analysis.* [Data on TVL and Liquidity Pool concentration].
*   **OECD (2022).** *Crypto-Asset Reporting Framework and Amendments to the Common Reporting Standard.* [Standard for automatic exchange of tax information].
*   **Zetzsche, D. A., Arner, D. W., & Buckley, R. P. (2020).** *Decentralized Finance: The Future of Financial Regulation.* University of Luxembourg Law Working Paper.

